
# OPTIMAL WEEKLY ROUTING WITH GOOGLE OR-TOOLS
Constraints:
- 20 employees working over 5 days (Monday-Friday)
- Max 3 clients per employee per day
- Max 6 hours work time per day (travel + care hours)
- Compatibility based on pets (dogs/cats) and smoking
- Interactive Map with day selection


# 1. Import Libraries

In [44]:
# !pip install ipywidgets

In [45]:
import pandas as pd
import numpy as np
import networkx as nx
from shapely import wkt
import folium
from scipy.spatial import cKDTree
from ortools.constraint_solver import routing_enums_pb2, pywrapcp
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta
from ipywidgets import interact, Dropdown
from IPython.display import IFrame, display, HTML

print('Bibliotheken geladen.')

Bibliotheken geladen.


# 2. Wegennet laden en graph bouwen

In [46]:
edges_df = pd.read_csv('../output/heerlen_edge_table.csv')
print(f'Aantal edges: {len(edges_df)}')
edges_df['geometry'] = edges_df['geometry'].apply(wkt.loads)

G = nx.Graph()
node_coords = {}
edge_geom = {}

for _, row in edges_df.iterrows():
    geom = row['geometry']
    coords = list(geom.coords)
    u, v = row['u'], row['v']
    G.add_edge(u, v, weight=row['travel_time_min'], geometry=geom)
    node_coords[u] = (coords[0][0], coords[0][1])
    node_coords[v] = (coords[-1][0], coords[-1][1])
    edge_geom[(u, v)] = geom
    edge_geom[(v, u)] = geom

print(f'Graph: {G.number_of_nodes()} knopen, {G.number_of_edges()} takken.')

node_ids = list(node_coords.keys())
node_lons_arr = np.array([node_coords[n][0] for n in node_ids])
node_lats_arr = np.array([node_coords[n][1] for n in node_ids])
kd_tree = cKDTree(np.column_stack((node_lons_arr, node_lats_arr)))

def nearest_node(lon, lat):
    _, idx = kd_tree.query([lon, lat])
    return node_ids[idx]

Aantal edges: 7183
Graph: 3120 knopen, 4340 takken.


# 3. Medewerkers laden (inclusief huisdier- en rookvoorkeuren)

In [47]:
employee_data = [
    ('employees 1',  50.8872, 5.9812),
    ('employees 2',  50.8895, 5.9820),
    ('employees 3',  50.8883, 5.9830),
    ('employees 4',  50.8855, 5.9795),
    ('employees 5',  50.8945, 5.9660),
    ('employees 6',  50.8878, 5.9808),
    ('employees 7',  50.8948, 5.9700),
    ('employees 8',  50.8870, 5.9825),
    ('employees 9',  50.8868, 5.9817),
    ('employees 10', 50.8785, 5.9750),
    ('employees 11', 50.8840, 5.9810),
    ('employees 12', 50.8860, 5.9835),
    ('employees 13', 50.8850, 5.9880),
    ('employees 14', 50.8890, 5.9822),
    ('employees 15', 50.8710, 5.9920),
    ('employees 16', 50.8810, 5.9680),
    ('employees 17', 50.8952, 5.9672),
    ('employees 18', 50.8875, 5.9805),
    ('employees 19', 50.8940, 5.9665),
    ('employees 20', 50.8790, 5.9760),
]
employees_df = pd.DataFrame(employee_data, columns=['name', 'lat', 'lon'])
employees_df['node'] = employees_df.apply(lambda r: nearest_node(r['lon'], r['lat']), axis=1)

emp_extra = pd.read_csv('../output/employees.csv')
emp_extra['name'] = emp_extra['name'].str.strip()
employees_df = employees_df.merge(emp_extra[['name', 'dogs', 'cats', 'smokes']], on='name', how='left')
employees_df['dogs'] = employees_df['dogs'].fillna(-1).astype(int)
employees_df['cats'] = employees_df['cats'].fillna(-1).astype(int)
employees_df['smokes'] = employees_df['smokes'].fillna(False).astype(bool)

print(f'Aantal medewerkers: {len(employees_df)}')

Aantal medewerkers: 20


# 4. Cliënten laden (coördinaten, zorgtijd, huisdieren, rook, tijdvenster)

In [48]:
clients_df = pd.read_csv('../output/clients.csv')

# Coördinaten detecteren
coord_col = None
for col in clients_df.columns:
    sample = clients_df[col].dropna().astype(str).iloc[0]
    parts = sample.replace(',', ' ').replace(';', ' ').split()
    if len(parts) == 2:
        try:
            float(parts[0]); float(parts[1])
            coord_col = col
            break
        except:
            pass
coord_col = coord_col or clients_df.columns[0]

def split_coords(s):
    parts = str(s).replace(';', ' ').replace(',', ' ').split()
    return (float(parts[0]), float(parts[1])) if len(parts) == 2 else (np.nan, np.nan)

clients_df[['lat', 'lon']] = clients_df[coord_col].apply(lambda x: pd.Series(split_coords(x)))
clients_df = clients_df.dropna(subset=['lat', 'lon']).reset_index(drop=True)
clients_df['client_id'] = clients_df.index
clients_df['node'] = clients_df.apply(lambda r: nearest_node(r['lon'], r['lat']), axis=1)

# Kolommen standaard invullen
for col in ['dogs', 'cats', 'smokes', 'care_hours']:
    if col not in clients_df.columns:
        clients_df[col] = 0 if col in ['dogs', 'cats'] else (False if col == 'smokes' else 1.0)
clients_df['smokes'] = clients_df['smokes'].astype(bool)
clients_df['care_hours'] = pd.to_numeric(clients_df['care_hours'], errors='coerce').fillna(1.0)

# Tijdvenster omzetten naar minuten na 07:00
def window_to_minutes(t_str):
    h, m = map(int, t_str.split(':'))
    return h * 60 + m - 420  # 07:00 = 0
clients_df['tw_min'] = clients_df['time_window_start'].apply(window_to_minutes)
clients_df['tw_max'] = clients_df['time_window_end'].apply(window_to_minutes)

print(f'Aantal cliënten: {len(clients_df)}')

Aantal cliënten: 100


# 5. Reistijdenmatrix berekenen

In [49]:
N_EMPLOYEES = len(employees_df)
N_CLIENTS = len(clients_df)
N_TOTAL = N_EMPLOYEES + N_CLIENTS   # 20 + 100 = 120
SCALE = 100

all_nodes = employees_df['node'].tolist() + clients_df['node'].tolist()
unique_sources = list(set(all_nodes))
print(f'Dijkstra uitvoeren vanaf {len(unique_sources)} unieke knopen ...')

dist_from = {}
for i, src in enumerate(unique_sources):
    dist_from[src] = nx.single_source_dijkstra_path_length(G, src, weight='weight')
    if (i+1) % 20 == 0:
        print(f'  {i+1}/{len(unique_sources)} gereed')

time_matrix = np.zeros((N_TOTAL, N_TOTAL), dtype=np.int64)
for i in range(N_TOTAL):
    src_graph = all_nodes[i]
    lengths = dist_from[src_graph]
    for j in range(N_TOTAL):
        dst_graph = all_nodes[j]
        t = lengths.get(dst_graph, float('inf'))
        time_matrix[i][j] = int(t * SCALE) if t != float('inf') else 10_000_000

print(f'Reistijdmatrix: {time_matrix.shape}')
print(f'Min reistijd: {time_matrix[time_matrix>0].min()/SCALE:.1f} min')
print(f'Max reistijd: {time_matrix[time_matrix<10_000_000].max()/SCALE:.1f} min')

Dijkstra uitvoeren vanaf 102 unieke knopen ...
  20/102 gereed
  40/102 gereed
  60/102 gereed
  80/102 gereed
  100/102 gereed
Reistijdmatrix: (120, 120)
Min reistijd: 0.0 min
Max reistijd: 11.2 min


# 6. Meerdaags model instellen (100 voertuigen: 20 medewerkers × 5 dagen)

In [50]:
NUM_DAYS = 5
MAX_CLIENTS_PER_VEHICLE = 3
MAX_WORK_MINUTES = 360  # 6 uur

vehicles = []
for emp_id in range(N_EMPLOYEES):
    for day in range(NUM_DAYS):
        vehicles.append({
            'vehicle_id': len(vehicles),
            'emp_id': emp_id,
            'day': day,
            'start_node': emp_id,
            'end_node': emp_id
        })
N_VEHICLES = len(vehicles)
print(f'Aantal voertuigen (medewerker×dag): {N_VEHICLES}')

Aantal voertuigen (medewerker×dag): 100


# 7. OR-Tools data model (tijdvensters, capaciteit, werktijdlimiet)

In [51]:
data = {}
data['time_matrix'] = time_matrix.tolist()
data['num_vehicles'] = N_VEHICLES
data['starts'] = [v['start_node'] for v in vehicles]
data['ends']   = [v['end_node'] for v in vehicles]
data['demands'] = [0] * N_EMPLOYEES + [1] * N_CLIENTS
data['capacities'] = [MAX_CLIENTS_PER_VEHICLE] * N_VEHICLES

# Zorgtijd in minuten (alleen voor cliënten, voor thuis = 0)
service_time = [0] * N_EMPLOYEES + (clients_df['care_hours'] * 60).round().astype(int).tolist()

# Tijdvensters: (start, end) in minuten na 07:00
time_windows = [(0, 660)] * N_EMPLOYEES  # thuis: 07:00 - 18:00
for _, client in clients_df.iterrows():
    time_windows.append((client['tw_min'], client['tw_max']))

manager = pywrapcp.RoutingIndexManager(
    len(data['time_matrix']),
    data['num_vehicles'],
    data['starts'],
    data['ends']
)
routing = pywrapcp.RoutingModel(manager)

# --- Callback voor reistijd + zorgtijd (zorgtijd wordt opgeteld bij vertrek van een knoop) ---
def total_time_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    travel = data['time_matrix'][from_node][to_node]           # al geschaald met SCALE
    service = service_time[from_node] * SCALE                  # zorgtijd ook schalen
    return travel + service

transit_callback = routing.RegisterTransitCallback(total_time_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback)

# --- Capaciteitsdimensie (max 3 cliënten per voertuig) ---
def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return data['demands'][from_node]

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index, 0, data['capacities'], True, 'Capacity'
)

# --- Tijdsdimensie (bevat reistijd + zorgtijd) ---
routing.AddDimension(
    transit_callback,          # transit callback met reistijd+zorgtijd
    0,                         # slack
    660 * SCALE,               # maximum totaal (18:00) – later per voertuig beperkt tot 6 uur
    False,                     # geen start slack
    'Time'
)
time_dim = routing.GetDimensionOrDie('Time')

# Tijdvensters instellen (arrival time moet binnen venster vallen)
for node in range(N_TOTAL):
    index = manager.NodeToIndex(node)
    tw_min, tw_max = time_windows[node]
    time_dim.CumulVar(index).SetRange(tw_min * SCALE, tw_max * SCALE)

# Maximale werktijd per voertuig instellen (360 minuten = 6 uur)
MAX_WORK_SCALED = MAX_WORK_MINUTES * SCALE
for v in range(N_VEHICLES):
    start_index = routing.Start(v)
    end_index = routing.End(v)
    # De totale tijd van start tot eind (inclusief alle reistijd en zorgtijd) mag niet > 360 minuten zijn
    time_dim.SetSpanUpperBoundForVehicle(MAX_WORK_SCALED, v)

print('Data model klaar (zorgtijd zit in de tijdsdimensie, max 6 uur per dag).')

Data model klaar (zorgtijd zit in de tijdsdimensie, max 6 uur per dag).


# 8. Compatibiliteit (huisdieren/rook) en voertuigbeperkingen

In [52]:
compatible_emp_per_client = []
skipped_clients = []
for cid in range(N_CLIENTS):
    client = clients_df.iloc[cid]
    compatible = []
    for emp_id, emp in employees_df.iterrows():
        if emp['dogs'] != -1 and client['dogs'] > emp['dogs']:
            continue
        if emp['cats'] != -1 and client['cats'] > emp['cats']:
            continue
        if not emp['smokes'] and client['smokes']:
            continue
        compatible.append(emp_id)
    if not compatible:
        skipped_clients.append(cid)
    compatible_emp_per_client.append(compatible)

solver = routing.solver()
for cid in range(N_CLIENTS):
    node_idx = manager.NodeToIndex(N_EMPLOYEES + cid)
    if cid in skipped_clients:
        routing.AddDisjunction([node_idx], 10_000_000)
    else:
        vehicle_var = routing.VehicleVar(node_idx)
        for v in range(N_VEHICLES):
            emp_id = vehicles[v]['emp_id']
            if emp_id not in compatible_emp_per_client[cid]:
                solver.Add(vehicle_var != v)

print(f'Cliënten zonder geschikte medewerker: {len(skipped_clients)}')

Cliënten zonder geschikte medewerker: 0


# 9. VRP oplossen

In [53]:
search_params = pywrapcp.DefaultRoutingSearchParameters()
search_params.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
)
search_params.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_params.time_limit.seconds = 180
search_params.log_search = True

print('Meerdaagse VRP wordt opgelost (max 3 cliënten/dag, max 6u/dag, tijdvensters)...')
solution = routing.SolveWithParameters(search_params)

if solution:
    print(f'\nOplossing gevonden! Totale reistijd: {solution.ObjectiveValue() / SCALE:.1f} min')
else:
    print('\nGeen oplossing gevonden.')

Meerdaagse VRP wordt opgelost (max 3 cliënten/dag, max 6u/dag, tijdvensters)...

Oplossing gevonden! Totale reistijd: 12478.9 min


# 10. Routes extraheren

In [54]:
def extract_routes_safe(solution, routing, manager):
    routes = []
    for v in range(N_VEHICLES):
        try:
            index = routing.Start(v)
            nodes = []
            # Verzamel nodes tot het einde
            while not routing.IsEnd(index):
                node = manager.IndexToNode(index)
                nodes.append(node)
                index = solution.Value(routing.NextVar(index))
            nodes.append(manager.IndexToNode(index))
            
            # Alleen cliënten (node >= N_EMPLOYEES)
            client_ids = [n - N_EMPLOYEES for n in nodes if n >= N_EMPLOYEES]
            if not client_ids:
                # Geen cliënten: voeg toch een lege route toe? Nee, overslaan
                continue
            
            # Haal totale werktijd uit de Time dimensie
            time_dim = routing.GetDimensionOrDie('Time')
            start_cumul = solution.Value(time_dim.CumulVar(routing.Start(v)))
            end_cumul = solution.Value(time_dim.CumulVar(routing.End(v)))
            work_time = (end_cumul - start_cumul) / SCALE
            
            # Reistijd (exclusief zorgtijd) apart berekenen (optioneel)
            travel_time = 0.0
            for i in range(len(nodes)-1):
                travel_time += time_matrix[nodes[i]][nodes[i+1]] / SCALE
            
            routes.append({
                'vehicle_id': v,
                'emp_id': vehicles[v]['emp_id'],
                'day': vehicles[v]['day'],
                'nodes': nodes,
                'client_ids': client_ids,
                'work_time': work_time,
                'travel_time': travel_time
            })
        except Exception as e:
            print(f"Waarschuwing: route voor voertuig {v} kon niet worden uitgelezen: {e}")
            continue
    return routes

if solution:
    all_routes = extract_routes_safe(solution, routing, manager)
    print(f'{len(all_routes)} routes geëxtraheerd (alleen routes met minimaal 1 cliënt).')
else:
    all_routes = []
    print('Geen oplossing beschikbaar.')

45 routes geëxtraheerd (alleen routes met minimaal 1 cliënt).


# 11. Overzicht van niet-ingeplande cliënten

In [55]:
if solution:
    geplande_clients = set()
    for r in all_routes:
        geplande_clients.update(r['client_ids'])
    alle_clients = set(range(N_CLIENTS))
    niet_gepland = alle_clients - geplande_clients
    if niet_gepland:
        print(f'\nNiet ingeplande cliënten ({len(niet_gepland)}):')
        for cid in sorted(niet_gepland):
            c = clients_df.iloc[cid]
            reden = 'geen geschikte medewerker' if cid in skipped_clients else 'capaciteit/tijd te krap'
            print(f'  Cliënt {cid} ({c["name"] if "name" in c else ""}): {reden}')
    else:
        print('\nAlle cliënten zijn ingepland!')


Alle cliënten zijn ingepland!


# 12. Samenvatting per medewerker per dag

In [56]:
if solution:
    print('\n=== Samenvatting per medewerker per dag ===')
    for day in range(NUM_DAYS):
        dag_str = ['maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag'][day]
        print(f'\n--- Dag {day+1} ({dag_str}) ---')
        dag_routes = [r for r in all_routes if r['day'] == day and r['client_ids']]
        for r in dag_routes:
            emp_name = employees_df.loc[r['emp_id'], 'name']
            print(f'{emp_name:15s}: {len(r["client_ids"])} cliënten | werktijd {r["work_time"]:.1f} min | reistijd {r["travel_time"]:.1f} min | stops: {r["client_ids"]}')


=== Samenvatting per medewerker per dag ===

--- Dag 1 (maandag) ---
employees 2    : 2 cliënten | werktijd 307.2 min | reistijd 7.2 min | stops: [54, 1]
employees 3    : 2 cliënten | werktijd 309.2 min | reistijd 9.2 min | stops: [79, 49]
employees 4    : 2 cliënten | werktijd 271.2 min | reistijd 1.2 min | stops: [64, 78]
employees 5    : 3 cliënten | werktijd 334.6 min | reistijd 4.6 min | stops: [97, 58, 63]
employees 6    : 1 cliënten | werktijd 90.0 min | reistijd 0.0 min | stops: [68]
employees 7    : 2 cliënten | werktijd 280.1 min | reistijd 10.1 min | stops: [67, 57]
employees 9    : 3 cliënten | werktijd 331.2 min | reistijd 1.2 min | stops: [21, 38, 32]
employees 11   : 1 cliënten | werktijd 152.2 min | reistijd 2.2 min | stops: [17]
employees 13   : 3 cliënten | werktijd 305.2 min | reistijd 5.2 min | stops: [23, 39, 71]
employees 14   : 2 cliënten | werktijd 301.1 min | reistijd 1.1 min | stops: [73, 42]
employees 15   : 3 cliënten | werktijd 337.6 min | reistijd 7.6 min

# 13. Interactieve kaart (dropdown voor dagen)

In [57]:
if solution:
    import base64
    from IPython.display import display, HTML

    EMPLOYEE_COLORS = [
        '#e6194b','#3cb44b','#ffe119','#4363d8','#f58231',
        '#911eb4','#42d4f4','#f032e6','#bfef45','#fabed4',
        '#469990','#dcbeff','#9A6324','#ff8c00','#800000',
        '#aaffc3','#808000','#00bfff','#000075','#808080',
    ]

    def nodes_to_latlon(path_nodes):
        latlon = []
        for i in range(len(path_nodes)-1):
            u, v = path_nodes[i], path_nodes[i+1]
            geom = edge_geom.get((u, v))
            if geom is None:
                cu = node_coords.get(u)
                cv = node_coords.get(v)
                if cu: latlon.append((cu[1], cu[0]))
                if cv: latlon.append((cv[1], cv[0]))
                continue
            coords = list(geom.coords)
            cu = node_coords.get(u)
            if cu and len(coords) >= 2:
                if abs(coords[-1][0] - cu[0]) < abs(coords[0][0] - cu[0]):
                    coords = coords[::-1]
            latlon.extend([(lat, lon) for lon, lat in coords])
        return latlon

    def road_segment(node_a, node_b):
        try:
            path = nx.shortest_path(G, source=node_a, target=node_b, weight='weight')
            return nodes_to_latlon(path)
        except nx.NetworkXNoPath:
            ca, cb = node_coords.get(node_a), node_coords.get(node_b)
            res = []
            if ca: res.append((ca[1], ca[0]))
            if cb: res.append((cb[1], cb[0]))
            return res

    def generate_day_map(day_index):
        day_name = ['Maandag', 'Dinsdag', 'Woensdag', 'Donderdag', 'Vrijdag'][day_index]
        day_routes = [r for r in all_routes if r['day'] == day_index and r['client_ids']]

        center_lat = float(np.mean(node_lats_arr))
        center_lon = float(np.mean(node_lons_arr))
        m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='CartoDB positron')

        for _, row in edges_df.iterrows():
            latlon = [(lat, lon) for lon, lat in row['geometry'].coords]
            folium.PolyLine(locations=latlon, color='#cccccc', weight=1, opacity=0.3).add_to(m)

        for r in day_routes:
            color = EMPLOYEE_COLORS[r['emp_id'] % len(EMPLOYEE_COLORS)]
            emp_name = employees_df.loc[r['emp_id'], 'name']
            nodes_seq = r['nodes']
            graph_seq = [all_nodes[n] for n in nodes_seq]
            for seg_i in range(len(graph_seq)-1):
                latlon = road_segment(graph_seq[seg_i], graph_seq[seg_i+1])
                if len(latlon) >= 2:
                    folium.PolyLine(
                        locations=latlon, color=color, weight=4, opacity=0.85,
                        tooltip=f'{emp_name} | dag {day_name}'
                    ).add_to(m)

            for stop_idx, cid in enumerate(r['client_ids']):
                client = clients_df.iloc[cid]
                folium.CircleMarker(
                    location=[client['lat'], client['lon']],
                    radius=6, color='white', weight=1.5, fill=True,
                    fill_color=color, fill_opacity=0.9,
                    popup=folium.Popup(
                        f'<b>{client["name"] if "name" in client else f"Cliënt {cid}"}</b><br>'
                        f'Medewerker: {emp_name}<br>Stop {stop_idx+1}',
                        max_width=200
                    ),
                    tooltip=f'{emp_name} stop {stop_idx+1}'
                ).add_to(m)

        for emp_id, emp in employees_df.iterrows():
            color = EMPLOYEE_COLORS[emp_id % len(EMPLOYEE_COLORS)]
            has_route = any(r['emp_id'] == emp_id for r in day_routes)
            popup_text = f"{emp['name']}<br>{'Wel actief' if has_route else 'Geen bezoeken'}"
            folium.Marker(
                location=[emp['lat'], emp['lon']],
                icon=folium.DivIcon(
                    html=f'<div style="width:22px;height:22px;background:{color};border:3px solid white;border-radius:50%;box-shadow:0 2px 6px rgba(0,0,0,.5);"></div>',
                    icon_size=(22,22), icon_anchor=(11,11)
                ),
                popup=folium.Popup(popup_text, max_width=240),
                tooltip=f"{emp['name']} (thuis)"
            ).add_to(m)

        legend_html = f'''
        <div style="position:fixed;bottom:20px;left:20px;z-index:1000;background:rgba(255,255,255,0.96);
                    padding:10px 14px;border-radius:8px;font-size:11px;font-family:sans-serif;
                    box-shadow:0 2px 10px rgba(0,0,0,.3);max-height:400px;overflow-y:auto;">
          <b>{day_name}</b><br>
          <span style="color:#777;">{len(day_routes)} actieve medewerkers</span>
          <table style="margin-top:6px;">'''
        for r in day_routes:
            emp_name = employees_df.loc[r['emp_id'], 'name']
            legend_html += f'<tr><td style="padding:2px;">•</td><td><b>{emp_name}</b></td><td style="padding-left:10px;">{len(r["client_ids"])} cliënten</td></tr>'
        legend_html += '</table></div>'
        m.get_root().html.add_child(folium.Element(legend_html))
        return m

    # Genereer HTML voor elke dag en sla op als string
    day_maps_html = []
    for day in range(5):
        m = generate_day_map(day)
        day_maps_html.append(m.get_root().render())

    # Bouw de overzichtspagina met tabs (geen externe bestanden)
    overview_html = f'''<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>VRP Routes per dag</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background: #f5f5f5;
        }}
        .tabs {{
            display: flex;
            gap: 10px;
            margin-bottom: 20px;
            flex-wrap: wrap;
        }}
        .tab-button {{
            padding: 10px 20px;
            background: #ddd;
            border: none;
            cursor: pointer;
            font-size: 16px;
            border-radius: 5px;
            transition: 0.3s;
        }}
        .tab-button:hover {{
            background: #bbb;
        }}
        .tab-button.active {{
            background: #007bff;
            color: white;
        }}
        .map-container {{
            width: 100%;
            height: 80vh;
            border: 1px solid #ccc;
            background: white;
        }}
        .map-iframe {{
            width: 100%;
            height: 100%;
            border: none;
        }}
    </style>
</head>
<body>
    <h1>Routes per dag - VRP planning</h1>
    <div class="tabs">
        <button class="tab-button active" onclick="showDay(0)">Maandag</button>
        <button class="tab-button" onclick="showDay(1)">Dinsdag</button>
        <button class="tab-button" onclick="showDay(2)">Woensdag</button>
        <button class="tab-button" onclick="showDay(3)">Donderdag</button>
        <button class="tab-button" onclick="showDay(4)">Vrijdag</button>
    </div>
    <div class="map-container">
        <iframe id="mapFrame" class="map-iframe" srcdoc="{day_maps_html[0].replace('"', '&quot;')}"></iframe>
    </div>
    <script>
        const maps = {day_maps_html};
        function showDay(dayIndex) {{
            const iframe = document.getElementById('mapFrame');
            iframe.srcdoc = maps[dayIndex].replace(/&quot;/g, '"');
            var buttons = document.getElementsByClassName('tab-button');
            for (var i = 0; i < buttons.length; i++) {{
                buttons[i].classList.remove('active');
            }}
            buttons[dayIndex].classList.add('active');
        }}
    </script>
</body>
</html>'''

    # Opslaan als enkel HTML-bestand
    with open('../output/routes_overview.html', 'w', encoding='utf-8') as f:
        f.write(overview_html)

    print('Overzichtspagina opgeslagen: ../output/routes_overview.html')
    display(HTML('<iframe src="../output/routes_overview.html" width="100%" height="700px" style="border:none;"></iframe>'))
else:
    print('Geen routes – geen kaart beschikbaar.')

Overzichtspagina opgeslagen: ../output/routes_overview.html


# 14. Dagplanning per medewerker (tekstueel)

In [60]:
if solution:
    EMPLOYEE_COLORS = [
        '#e6194b','#3cb44b','#ffe119','#4363d8','#f58231',
        '#911eb4','#42d4f4','#f032e6','#bfef45','#fabed4',
        '#469990','#dcbeff','#9A6324','#ff8c00','#800000',
        '#aaffc3','#808000','#00bfff','#000075','#808080',
    ]

    def nodes_to_latlon(path_nodes):
        latlon = []
        for i in range(len(path_nodes)-1):
            u, v = path_nodes[i], path_nodes[i+1]
            geom = edge_geom.get((u, v))
            if geom is None:
                cu = node_coords.get(u)
                cv = node_coords.get(v)
                if cu: latlon.append((cu[1], cu[0]))
                if cv: latlon.append((cv[1], cv[0]))
                continue
            coords = list(geom.coords)
            cu = node_coords.get(u)
            if cu and len(coords) >= 2:
                if abs(coords[-1][0] - cu[0]) < abs(coords[0][0] - cu[0]):
                    coords = coords[::-1]
            latlon.extend([(lat, lon) for lon, lat in coords])
        return latlon

    def road_segment(node_a, node_b):
        try:
            path = nx.shortest_path(G, source=node_a, target=node_b, weight='weight')
            return nodes_to_latlon(path)
        except nx.NetworkXNoPath:
            ca, cb = node_coords.get(node_a), node_coords.get(node_b)
            res = []
            if ca: res.append((ca[1], ca[0]))
            if cb: res.append((cb[1], cb[0]))
            return res

    def draw_day(day_index):
        day_name = ['Maandag', 'Dinsdag', 'Woensdag', 'Donderdag', 'Vrijdag'][day_index]
        day_routes = [r for r in all_routes if r['day'] == day_index and r['client_ids']]

        center_lat = float(np.mean(node_lats_arr))
        center_lon = float(np.mean(node_lons_arr))
        m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='CartoDB positron')

        for _, row in edges_df.iterrows():
            latlon = [(lat, lon) for lon, lat in row['geometry'].coords]
            folium.PolyLine(locations=latlon, color='#cccccc', weight=1, opacity=0.3).add_to(m)

        for r in day_routes:
            color = EMPLOYEE_COLORS[r['emp_id'] % len(EMPLOYEE_COLORS)]
            emp_name = employees_df.loc[r['emp_id'], 'name']
            nodes_seq = r['nodes']
            graph_seq = [all_nodes[n] for n in nodes_seq]
            for seg_i in range(len(graph_seq)-1):
                latlon = road_segment(graph_seq[seg_i], graph_seq[seg_i+1])
                if len(latlon) >= 2:
                    folium.PolyLine(
                        locations=latlon, color=color, weight=4, opacity=0.85,
                        tooltip=f'{emp_name} | dag {day_name}'
                    ).add_to(m)

            for stop_idx, cid in enumerate(r['client_ids']):
                client = clients_df.iloc[cid]
                folium.CircleMarker(
                    location=[client['lat'], client['lon']],
                    radius=6, color='white', weight=1.5, fill=True,
                    fill_color=color, fill_opacity=0.9,
                    popup=folium.Popup(
                        f'<b>{client["name"] if "name" in client else f"Cliënt {cid}"}</b><br>'
                        f'Medewerker: {emp_name}<br>Stop {stop_idx+1}',
                        max_width=200
                    ),
                    tooltip=f'{emp_name} stop {stop_idx+1}'
                ).add_to(m)

        for emp_id, emp in employees_df.iterrows():
            color = EMPLOYEE_COLORS[emp_id % len(EMPLOYEE_COLORS)]
            has_route = any(r['emp_id'] == emp_id for r in day_routes)
            popup_text = f"{emp['name']}<br>{'Wel actief' if has_route else 'Geen bezoeken'}"
            folium.Marker(
                location=[emp['lat'], emp['lon']],
                icon=folium.DivIcon(
                    html=f'<div style="width:22px;height:22px;background:{color};border:3px solid white;border-radius:50%;box-shadow:0 2px 6px rgba(0,0,0,.5);"></div>',
                    icon_size=(22,22), icon_anchor=(11,11)
                ),
                popup=folium.Popup(popup_text, max_width=240),
                tooltip=f"{emp['name']} (thuis)"
            ).add_to(m)

        legend_html = f'''
        <div style="position:fixed;bottom:20px;left:20px;z-index:1000;background:rgba(255,255,255,0.96);
                    padding:10px 14px;border-radius:8px;font-size:11px;font-family:sans-serif;
                    box-shadow:0 2px 10px rgba(0,0,0,.3);max-height:400px;overflow-y:auto;">
          <b>{day_name}</b><br>
          <span style="color:#777;">{len(day_routes)} actieve medewerkers</span>
          <table style="margin-top:6px;">'''
        for r in day_routes:
            emp_name = employees_df.loc[r['emp_id'], 'name']
            legend_html += f'<tr><td style="padding:2px;">•</td><td><b>{emp_name}</b></td><td style="padding-left:10px;">{len(r["client_ids"])} cliënten</td></tr>'
        legend_html += '</table></div>'
        m.get_root().html.add_child(folium.Element(legend_html))
        return m

    # Mapping van dagindex naar Engelstalige bestandsnaam (route_monday.html etc.)
    day_names_nl = ['Maandag', 'Dinsdag', 'Woensdag', 'Donderdag', 'Vrijdag']
    day_names_en = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday']
    
    # Genereer kaarten voor alle dagen
    for day in range(5):
        m = draw_day(day)
        filename = f'../output/route_{day_names_en[day]}.html'
        m.save(filename)
        print(f'Kaart voor {day_names_nl[day]} opgeslagen: {filename}')

    # Maak een overzichtspagina met tabs (verwijst naar de nieuwe bestanden)
    html_content = f'''<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>VRP Routes per dag</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background: #f5f5f5;
        }}
        .tabs {{
            display: flex;
            gap: 10px;
            margin-bottom: 20px;
            flex-wrap: wrap;
        }}
        .tab-button {{
            padding: 10px 20px;
            background: #ddd;
            border: none;
            cursor: pointer;
            font-size: 16px;
            border-radius: 5px;
            transition: 0.3s;
        }}
        .tab-button:hover {{
            background: #bbb;
        }}
        .tab-button.active {{
            background: #007bff;
            color: white;
        }}
        .map-container {{
            width: 100%;
            height: 80vh;
            border: 1px solid #ccc;
            background: white;
        }}
        iframe {{
            width: 100%;
            height: 100%;
            border: none;
        }}
    </style>
</head>
<body>
    <h1>Routes per dag - VRP planning</h1>
    <div class="tabs">
'''
    for i, name_nl in enumerate(day_names_nl):
        active = 'active' if i == 0 else ''
        html_content += f'        <button class="tab-button {active}" onclick="showDay({i})">{name_nl}</button>\n'
    html_content += '''    </div>
    <div class="map-container">
        <iframe id="mapFrame" src="route_monday.html"></iframe>
    </div>
    <script>
        const dayFiles = ['''
    for en in day_names_en:
        html_content += f"'route_{en}.html', "
    html_content = html_content.rstrip(', ') + '''];

        function showDay(dayIndex) {
            document.getElementById('mapFrame').src = dayFiles[dayIndex];
            var buttons = document.getElementsByClassName('tab-button');
            for (var i = 0; i < buttons.length; i++) {
                buttons[i].classList.remove('active');
            }
            buttons[dayIndex].classList.add('active');
        }
    </script>
</body>
</html>'''

    with open('../output/routes_overview.html', 'w', encoding='utf-8') as f:
        f.write(html_content)
    print('Overzichtspagina opgeslagen: ../output/routes_overview.html')

    from IPython.display import IFrame, display
    display(IFrame(src='../output/routes_overview.html', width='100%', height=700))
else:
    print('Geen routes – geen kaart beschikbaar.')

Kaart voor Maandag opgeslagen: ../output/route_monday.html
Kaart voor Dinsdag opgeslagen: ../output/route_tuesday.html
Kaart voor Woensdag opgeslagen: ../output/route_wednesday.html
Kaart voor Donderdag opgeslagen: ../output/route_thursday.html
Kaart voor Vrijdag opgeslagen: ../output/route_friday.html
Overzichtspagina opgeslagen: ../output/routes_overview.html
